# Tutorial 3 — WorldView-3, Los Peñasquitos Canyon, three images blended

> Reading this on the docs site? To run it, [open a Codespace](https://stereopipeline-quickstart.readthedocs.io/en/latest/start/codespaces.html): ASP and `asp-plot` are already installed there.

[Tutorial 2](02_worldview_ucsd.ipynb) made a DEM from one pair of images. With a third image of the same place there are three pairs, and three DEMs that each miss different spots and carry different noise. Averaging them gives one DEM with fewer holes and less noise; on this image stack, that beat any single pair. This tutorial runs stereo on all three pairs of the same 1.2 × 1 km area and blends the results, in about 22 minutes after Tutorial 2 (about 25 on its own).

> To run it, click **Run All** at the top, then scroll down to follow it: each step shows its progress while it runs, then its results.

| Image | Date | Looking |
|---|---|---|
| `104001000496A100` | 2014-11-09 | 1.1° from straight down |
| `10400100071D8800` | 2015-01-24 | 16.3° to the side |
| `10400100057DD500` | 2014-12-23 | 24.1° to the side |

The steps:

1. Download the three images (two are reused from Tutorial 2).
2. Look at the viewing geometry.
3. Orthorectify all three (two are reused from Tutorial 2).
4. Refine the three cameras together (bundle adjustment).
5. Run stereo matching on each pair, for three DEMs.
6. Blend the three DEMs into one.
7. Make a PDF report of the blended DEM, with a comparison against ICESat-2.
8. Download the results.

In [ ]:
from pathlib import Path

from utils import run, run_stereo, timer

DATA = "../data/ucsd_penasquitos"
LOGS = f"{DATA}/logs"

!mkdir -p {DATA}

# The study area, xmin ymin xmax ymax in UTM zone 11N (EPSG:32611) meters:
# 1200 × 1000 m at the west end of Los Peñasquitos Canyon, as in Tutorial 2.
T_SRS = "EPSG:32611"
T_PROJWIN = "480420 3641720 481620 3642720"

# Pixel size of the orthorectified images (3840 × 3200 pixels, four
# 1920 × 1600 stereo tiles) and of the DEMs, in meters.
GSD = 0.3125
TILE_W, TILE_H = 1920, 1600
DEM_RES = 1.0

# The Copernicus reference DEM, fetched by the download step.
REF_DEM = f"{DATA}/ref/cop30_penasquitos.tif"

SCENES = ["104001000496A100", "10400100071D8800", "10400100057DD500"]
PAIRS = [(SCENES[0], SCENES[1]), (SCENES[0], SCENES[2]), (SCENES[2], SCENES[1])]

FRESH_ICESAT2 = False
SHIPPED_SELECTIONS = f"{DATA}/icesat2_shipped/blend_figure_selections.yml"

## 1. Download the images

The same images as Tutorial 2 plus a third; images already downloaded are skipped. They are IARPA CORE3D data, released for public research; publications that use them should cite Hagstrom et al. (2021), "Cumulative assessment for urban 3D modeling" (IEEE IGARSS).

> Under a minute after Tutorial 2, or about 2 minutes without it.

In [ ]:
run(f"bash ../scripts/download_worldview_ucsd.sh {DATA} {' '.join(SCENES)}", log=f"{LOGS}/download.log", label="Downloading the images")

## 2. Viewing geometry

[More on viewing geometry](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/stereo-photogrammetry.html#viewing-geometry).

> About half a minute.

In [ ]:
with timer("Plotting the stereo geometry"):
    import contextily
    from asp_plot.stereo_geometry import StereoGeometryPlotter

    # The map tiles were fetched when the Codespace was built
    # (scripts/prefetch_basemap.py); without them, this fetches them itself.
    contextily.set_cache_dir("../data/basemap_cache")

    geometry = StereoGeometryPlotter(inputs=[f"{DATA}/{catid}_P001.xml" for catid in SCENES])
    geometry.stereo_geom_plot()

The three view directions spread around the sky: one near straight down and two from different sides. Each pair sees the ground from a different angle, so a spot hidden from one pair is often visible to another.

## 3. Orthorectify

The same step as Tutorial 2, for all three images: `mapproject` reads the study area out of each full image and drapes it over the reference DEM at 0.3125 m per pixel. Images already done are skipped.

> About half a minute per image.

In [ ]:
Path(f"{DATA}/ortho").mkdir(exist_ok=True)
for catid in SCENES:
    out = f"{DATA}/ortho/{catid}.tif"
    if Path(out).exists():
        print(f"{out} exists — skipping mapproject. Delete it to reprocess.")
    else:
        run(f"""
            mapproject -t dg --threads 4 --t_srs {T_SRS} --tr {GSD} --t_projwin {T_PROJWIN} \
                {REF_DEM} {DATA}/{catid}_P001.NTF {DATA}/{catid}_P001.xml {out}
            """,
            log=f"{LOGS}/mapproject_{catid}.log",
            label=f"Orthorectifying {catid}",
        )

!ls -lh {DATA}/ortho/*.tif

## 4. Bundle adjustment

The vendor's camera descriptions are good to a few meters, and the three don't quite agree with each other: a DEM from one pair sits several meters above or below a DEM from another, and averaging them as they are would smear every edge. ASP's `bundle_adjust` finds features that appear in two or three of the orthorectified images, traces them back through the cameras, and nudges the cameras until each feature's rays meet at one point. [What bundle adjustment does](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/bundle-adjustment.html).

> About half a minute.

In [ ]:
if Path(f"{DATA}/ba3/run-final_residuals_pointmap.csv").exists():
    print(f"{DATA}/ba3/run-final_residuals_pointmap.csv exists — skipping bundle_adjust. Delete {DATA}/ba3/ to reprocess.")
else:
    images = " ".join(f"{DATA}/{c}_P001.NTF" for c in SCENES)
    cameras = " ".join(f"{DATA}/{c}_P001.xml" for c in SCENES)
    orthos = " ".join(f"{DATA}/ortho/{c}.tif" for c in SCENES)
    run(f"""
        bundle_adjust -t dg --threads 4 --ip-per-tile 300 --matches-per-tile 150 \
            --mapprojected-data "{orthos} {REF_DEM}" \
            {images} {cameras} -o {DATA}/ba3/run
        """,
        log=f"{LOGS}/bundle_adjust_3.log",
        label="Bundle adjusting the three cameras",
    )

!cat {DATA}/ba3/run-initial_residuals_stats.txt {DATA}/ba3/run-final_residuals_stats.txt

In [ ]:
from asp_plot.bundle_adjust import ReadBundleAdjustFiles, PlotBundleAdjustFiles

ba_reader = ReadBundleAdjustFiles(DATA, "ba3")
initial_gdf, final_gdf = ba_reader.get_initial_final_residuals_gdfs()
PlotBundleAdjustFiles(
    [initial_gdf, final_gdf], lognorm=True, title="Bundle adjustment residuals",
).plot_n_gdfs(column_name="mean_residual", cbar_label="Mean residual (px)")

Each dot is a feature found in two or three images, colored by how far apart its projections land in the images (the reprojection error, in pixels). Before adjustment (left), the median is 3 to 5 pixels, which at 0.3 m per pixel is the few-meter disagreement between the cameras. After (right), it is about 0.3 pixels: the three cameras now agree. The adjustment moves the cameras relative to each other only; the whole set can still sit a few meters off the true ground, which the report's comparison with ICESat-2 measures.

## 5. Stereo matching, pair by pair

The same stereo as Tutorial 2, three times, with the adjusted cameras. For each pair, the cell shows the matched features, then the map of the area with the offsets between the images and the heights, as it fills in. [How matching works](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/stereo-photogrammetry.html).

> About 20 minutes, 6 to 7 minutes per pair.

In [ ]:
def pair_dir(left, right):
    return f"{DATA}/pairs/{left}_{right}"

for n, (left, right) in enumerate(PAIRS, start=1):
    out = pair_dir(left, right)
    if Path(f"{out}/run-DEM.tif").exists():
        print(f"{out}/run-DEM.tif exists — skipping. Delete {out}/ to reprocess.")
        continue
    run_stereo(f"""
        parallel_stereo \
            --stereo-algorithm asp_mgm --subpixel-mode 9 --cost-mode 3 \
            --job-size-w {TILE_W} --job-size-h {TILE_H} \
            --processes 4 --threads-multiprocess 1 --threads-singleprocess 4 \
            --bundle-adjust-prefix {DATA}/ba3/run \
            {DATA}/ortho/{left}.tif {DATA}/ortho/{right}.tif \
            {DATA}/{left}_P001.xml {DATA}/{right}_P001.xml \
            {out}/run {REF_DEM}
        """,
        prefix=f"{out}/run",
        log=f"{LOGS}/parallel_stereo_pair{n}.log",
        label=f"Stereo, pair {n} of 3",
    )
    run(
        f"point2dem --threads 4 --tr {DEM_RES} --t_srs {T_SRS} --errorimage {out}/run-PC.tif",
        log=f"{LOGS}/point2dem_pair{n}.log",
        label=f"Gridding pair {n} into a DEM",
    )

For each pair: the matched features (red), then the offset between the two input images at every pixel, split into its east-west and north-south parts, and the heights triangulated from them (right). The offsets are the [relief displacement](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/orthorectification.html) left after orthorectification: a few pixels for the nadir and 16° pair, more for the pairs with the 24° image, which looks further from straight down.

In [ ]:
from utils import show_hillshades

show_hillshades([f"{pair_dir(l, r)}/run-DEM.tif" for l, r in PAIRS],
                ["nadir + 16°", "nadir + 24°", "24° + 16°"])

The three pair DEMs as hillshades. They agree on the landforms, but each has its own holes (white) and its own speckle. The holes are mostly in the trees along the creek, where the rays from the two cameras meet badly. With the adjusted cameras the rays meet to within a few centimeters everywhere else, so `point2dem`'s outlier filter, which scales with that error, drops the tree points that Tutorial 2's unadjusted run kept; each pair drops different ones.

## 6. Blend the three DEMs

ASP's `dem_mosaic` averages the three DEMs pixel by pixel. Where one DEM has a hole, the others fill it.

> A few seconds.

In [ ]:
BLEND = f"{DATA}/pairs/{PAIRS[0][0]}_{PAIRS[0][1]}/run-blend-DEM.tif"
if Path(BLEND).exists():
    print(f"{BLEND} exists — skipping dem_mosaic. Delete it to reblend.")
else:
    dems = " ".join(f"{pair_dir(l, r)}/run-DEM.tif" for l, r in PAIRS)
    run(f"dem_mosaic --threads 4 {dems} -o {BLEND}", log=f"{LOGS}/dem_mosaic.log", label="Blending the three DEMs")

show_hillshades([f"{pair_dir(*PAIRS[0])}/run-DEM.tif", BLEND],
                ["One pair (nadir + 16°)", "Three pairs blended"])

Left, one pair; right, the blend. The blend fills most of the holes and the speckle averages out, while the spurs, the gullies and the track along the creek stay sharp.

## 7. Report

`asp_report`, from `asp-plot`, writes a PDF with everything about the run, here with the blended DEM as its result: the images, the viewing geometry, the bundle adjustment, the matches, the DEM, and how it compares with the reference map and with [ICESat-2](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/alignment.html), a satellite that measures heights directly with a laser, then shifted onto the ICESat-2 points with `pc_align`, since bundle adjustment only moved the cameras relative to each other. [What's in the report](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/visualization.html). Three of its figures appear below.

> About a minute.

In [ ]:
from utils import show_report_figures

STEREO = f"pairs/{PAIRS[0][0]}_{PAIRS[0][1]}"
FIGS = f"{DATA}/{STEREO}/report_figures"
if Path(f"{DATA}/{STEREO}/penasquitos_wv3_blend_report.pdf").exists():
    print(f"{DATA}/{STEREO}/penasquitos_wv3_blend_report.pdf exists — skipping asp_report. Delete it to regenerate.")
else:
    reuse = "" if FRESH_ICESAT2 else f"--reuse-selections {SHIPPED_SELECTIONS}"
    run(f"""
        asp_report \
            --directory {DATA} \
            --bundle-adjust-prefix ba3 \
            --stereo-directory {STEREO} \
            --dem-filename run-blend-DEM.tif \
            --reference-dem {REF_DEM} \
            --subset-km 0.25 \
            --pc-align-min-points 150 \
            {reuse} \
            --figures-directory {FIGS} \
            --report-filename penasquitos_wv3_blend_report.pdf
        """,
        log=f"{LOGS}/asp_report_blend.log",
        label="Making the report",
    )

show_report_figures(FIGS)

## 📄 [Open the report (PDF)](../data/ucsd_penasquitos/pairs/104001000496A100_10400100071D8800/penasquitos_wv3_blend_report.pdf)

It opens in a tab here. To keep a copy, right-click the file in the explorer on the left and choose **Download**. Compare it with Tutorial 2's report: the ICESat-2 profile follows the same track.

## 8. Download the results

The cell below puts the results into one zip file, `penasquitos_blend_results.zip`, in the top folder of the file explorer: the report, the blended DEM and its hillshade. Each file is named by place, sensor and dates, and the GeoTIFFs are Cloud Optimized GeoTIFFs with overviews, so they open quickly in QGIS, ArcGIS or any other GIS. A QGIS project file in the zip opens them at once. When the zip is ready, a notification at the bottom right points to it; right-click it in the explorer and choose **Download**.

> A few seconds.

Each pair's DEM and map of triangulation error stay in `data/ucsd_penasquitos/pairs/`; they can be downloaded one by one the same way.

In [ ]:
from utils import multidirectional_hillshade, zip_results

# Each result's file, its name in the zip (what it is, where, when, from what), and its kind.
NAME = "penasquitos_wv3_3pair_blend_20141109_20150124"
multidirectional_hillshade(BLEND, f"{DATA}/{STEREO}/run-blend-DEM_hs.tif")
zip_results(
    "../penasquitos_blend_results.zip",
    [
        (f"{DATA}/{STEREO}/penasquitos_wv3_blend_report.pdf", f"{NAME}_report.pdf", "pdf"),
        (BLEND, f"{NAME}_dem_1m.tif", "dem"),
        (f"{DATA}/{STEREO}/run-blend-DEM_hs.tif", f"{NAME}_dem_1m_hillshade.tif", "hillshade"),
    ],
    title=NAME,
)

### How to download it

The zip is in the **file explorer**, the panel on the left of the window with `data/`, `notebooks/` and the zip (if the panel is hidden, click the top icon in the bar at the far left). Right-click the zip there and choose **Download**; your browser then saves it like any download.

![The end of Tutorial 1: (1) the report link, (2) the results line, (3) the zip in the file explorer](https://stereopipeline-quickstart.readthedocs.io/en/latest/_images/launch-done.png)

The first time you right-click in the explorer, your browser may ask whether this page can see your clipboard. Click **Allow** or **Block**, whichever you prefer; the menu with **Download** works either way.

![The browser's clipboard question, with Block and Allow buttons](https://stereopipeline-quickstart.readthedocs.io/en/latest/_images/clipboard-prompt.png)